In [4]:
from dotenv import load_dotenv
import os



load_dotenv()

api_key = os.getenv('DEEPSEEK_API_KEY')
print(api_key)

sk-afe8802094354689a4c968accb7ff264


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage

checkpointer = InMemorySaver()

middleware = SummarizationMiddleware(
    model="deepseek-flash",
    trigger=("messages",3),
    keep=("messages",1),
)

agent = create_agent(
    model="deepseek-v4-pro",
    middleware=[middleware],
    checkpointer=checkpointer,
)

config:RunnableConfig={"configurable":{"thread_id":"thread_1"}}

agent.invoke({"messages":[HumanMessage(content="你好，我是奶龙")]},config)
agent.invoke({"messages":[HumanMessage(content="我最喜欢的运动是功夫")]},config)
agent.invoke({"messages":[HumanMessage(content="我喜欢捧腹大笑")]},config)

test_response=agent.invoke({"messages":[HumanMessage(content="你还记得我吗")]},config)

In [4]:
print(test_response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n用户（自称"奶龙"，中文交流）处于轻松闲聊阶段，尚未提出具体技术任务或工作请求。已分享个人兴趣：最喜欢的运动是**功夫（Kung Fu）**，以及**喜欢捧腹大笑**。当前目标是在友好互动中了解用户兴趣，等待用户明确后续需求。\n\n## SUMMARY\n\n- 用户身份：自称"奶龙"，使用中文交流；后续应始终用中文回复。\n- 用户已分享兴趣：\n  1. 最喜欢的运动：**功夫**。\n  2. 喜欢**捧腹大笑**（偏好欢乐/搞笑内容）。\n- AI 此前已就"功夫"热情回应（🦸\u200d♂️），并提出三个引导问题：是否练过功夫、传统武术还是现代搏击方向、最喜欢的功夫明星。\n- 对"捧腹大笑"，AI 已回应并追问：平时爱看搞笑视频、喜剧电影，还是喜欢听段子/冷笑话？并提议可讲一个"奶龙专属"小笑话。\n- 尚未确定任何具体任务、技术方案或关键决策。\n- 无被拒绝或放弃的选项。\n- 语气基调：友好、乐于助人、轻松活泼、幽默。\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- 等待用户回答：偏好哪种搞笑形式（搞笑视频 / 喜剧电影 / 段子 / 冷笑话），以及对功夫相关问题的回复。\n- 如用户接受，准备并讲一个"奶龙专属"小笑话。\n- 保持中文交流与友好活泼幽默的语气，围绕功夫与搞笑兴趣继续展开话题。\n- 在用户明确提出具体任务或需求后，再确定工作目标与执行方案。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='c165d068-6f7a-470d-89fb-7f605819bf05'), HumanMessage(content='你还记得我吗', additional_kwargs={}, response_metadata={}, id='1019931f-f1b5-4723-8ea7-6eff69833d86'), AIMessage(content='当然记得你！你是**奶龙**嘛

In [24]:
from langchain.chat_models import init_chat_model
import os
#初始化模型
model=init_chat_model(
    model="qwen3.8-flash",
    model_provider="openai",
    base_url=os.getenv("QWEN_API_URL"),
    api_key=os.getenv("QWEN_API_KEY"),
)

In [25]:
from langchain_tavily import TavilySearch
#定义工具
web_search = TavilySearch(
    max_results=5,
    topic="general"
)

In [26]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
#添加记忆管理，这里用数据库来存 
connection = sqlite3.connect('resources/personal_cook_master.db', check_same_thread=False)
checkpointer=SqliteSaver(connection)
checkpointer.setup()

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.runnables import RunnableConfig

middleware = SummarizationMiddleware(
    model="deepseek-flash",
    trigger=("messages",3),
    keep=("messages",1),
)

system_promt="你是一个私人厨师，收到用户提供的食材照片或清单后，给出健康美味的料理，规定必须用web_search查阅并给出示例图片，实在找不到可以自由发挥",

agent = create_agent(
    model=model,
    middleware=[middleware],
    system_prompt=system_promt,
    tools=[web_search],
    
    checkpointer=checkpointer,
)


In [28]:
from langchain_core.messages import HumanMessage

mutimodel_message=HumanMessage([
    {"type":"text","text":"帮我看看能做什么"},
    {"type":"image","url":"https://assets.699pic.com/public/web/images/600/580/239.jpg!seo.v1"},
])

config:RunnableConfig={"configurable":{"thread_id":"thread_3"}}

response=agent.invoke({"messages":[mutimodel_message]},config)

In [29]:
#友好打印
for massage in response['messages']:
    massage.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

Previous conversation was too long to summarize.
================================== Ai Message ==================================

已找到几套靠谱的做法和参考图，再补两份"耐放食材"和"水果+酸奶"的方案。
Tool Calls:
  tavily_search (call_2ba0556095f14d388f8ca978)
 Call ID: call_2ba0556095f14d388f8ca978
  Args:
    query: 土豆 洋葱 番茄 浓汤 健康食谱 少油 做法 参考图
    include_images: True
    search_depth: advanced
  tavily_search (call_3909ae892aac420680ee6cd9)
 Call ID: call_3909ae892aac420680ee6cd9
  Args:
    query: 苹果 酸奶 羽衣甘蓝 青提 思慕雪 bowl 食谱 做法
    include_images: True
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "土豆 洋葱 番茄 浓汤 健康食谱 少油 做法 参考图", "follow_up_questions": null, "answer": null, "images": ["https://i2.chuimg.com/2200daa66df24222a4bb6e166e178ba0_1080w_864h.jpg?imageView2/1/w/640/h/520/q/75/format/jpg", "ht